<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Estimate uncertainty

**[C03-M05-L02 · Estimate uncertainty](https://learning.nextia-ai.com/courses/math/m05/estimate-uncertainty/)** · C03, Essential Mathematics and Statistics for AI · Module 5, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** draw many samples from a known population of 6,000 reply times and count how many 95% intervals contain the true mean. Then calculate the standard error and the 95% interval for the 12 Monday reply times and for model B's accuracy, and build a bootstrap interval.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | A test set is a sample, and random samples scatter around the true value, from [Samples and populations: See it: random samples and a biased sample](https://learning.nextia-ai.com/courses/math/m05/samples-and-populations/#see-it-random-samples-and-a-biased-sample). The SD and the 12 Monday reply times, from [Describe a dataset: Spread: range, variance and standard deviation](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/#spread-range-variance-and-standard-deviation). The normal curve and its 95% rule, from [Distributions in practice: Normal: the bell-shaped curve](https://learning.nextia-ai.com/courses/math/m03/distributions-in-practice/#normal-the-bell-shaped-curve). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m05/estimate-uncertainty/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M05-L02-estimate-uncertainty/estimate-uncertainty-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the population of 6,000 reply times

The next cell defines `population`: the first-reply times, in minutes, of 6,000 made-up past tickets (about five weeks). It is the same population as in the lesson's explorer. Because you have all of it, you know the true mean.

Run the cell. You should see `Ready: 6000 tickets, true mean 26.8 minutes`.

In [ ]:
# The data, written in full (you do not need to read it).
population = [
    24, 9, 11, 32, 15, 48, 26, 33, 20, 11, 43, 44, 20, 11, 56, 14, 33, 60, 15, 20,
    24, 28, 14, 42, 22, 34, 35, 10, 13, 24, 31, 35, 15, 19, 18, 38, 14, 6, 27, 17,
    14, 14, 23, 8, 12, 26, 76, 20, 19, 15, 29, 40, 25, 38, 27, 51, 14, 30, 17, 15,
    24, 27, 90, 15, 23, 11, 39, 24, 24, 130, 13, 6, 8, 13, 21, 29, 20, 66, 12, 21,
    18, 14, 11, 10, 27, 30, 32, 13, 46, 38, 60, 9, 18, 51, 10, 11, 28, 16, 7, 26,
    26, 35, 72, 24, 15, 13, 20, 42, 92, 17, 62, 37, 13, 11, 6, 13, 12, 24, 22, 70,
    10, 6, 13, 15, 23, 24, 11, 11, 73, 27, 93, 44, 19, 45, 10, 19, 73, 5, 16, 29,
    30, 18, 47, 13, 58, 46, 10, 13, 20, 17, 19, 41, 10, 19, 16, 52, 19, 35, 33, 14,
    57, 42, 14, 29, 18, 15, 29, 29, 11, 23, 17, 13, 65, 12, 25, 18, 16, 9, 41, 8,
    23, 25, 51, 22, 14, 25, 30, 24, 31, 21, 35, 8, 55, 24, 22, 10, 10, 30, 26, 17,
    27, 38, 43, 14, 111, 9, 14, 41, 12, 42, 22, 32, 28, 38, 37, 26, 18, 29, 19, 12,
    18, 64, 23, 23, 27, 9, 31, 10, 20, 42, 19, 11, 81, 17, 13, 11, 18, 51, 10, 18,
    15, 11, 12, 30, 6, 47, 27, 21, 16, 15, 27, 34, 41, 78, 20, 61, 12, 29, 19, 56,
    24, 68, 28, 20, 51, 13, 24, 13, 18, 30, 48, 18, 19, 9, 31, 27, 17, 32, 21, 14,
    23, 13, 29, 52, 40, 60, 16, 41, 37, 118, 20, 17, 37, 24, 46, 69, 13, 14, 38, 30,
    10, 32, 12, 13, 13, 36, 18, 15, 11, 23, 9, 22, 29, 8, 17, 6, 13, 10, 10, 26,
    22, 24, 27, 18, 9, 10, 16, 32, 24, 93, 23, 22, 29, 22, 20, 28, 13, 31, 40, 23,
    15, 56, 81, 58, 19, 18, 33, 12, 42, 22, 12, 67, 20, 21, 9, 26, 72, 34, 46, 11,
    25, 28, 26, 19, 19, 20, 33, 9, 27, 18, 45, 21, 10, 15, 7, 56, 7, 51, 129, 26,
    15, 16, 24, 7, 15, 21, 48, 61, 5, 29, 40, 66, 23, 62, 19, 11, 27, 53, 28, 13,
    9, 11, 104, 24, 12, 18, 18, 20, 10, 27, 16, 11, 10, 37, 8, 21, 18, 165, 27, 34,
    40, 17, 29, 57, 13, 45, 20, 18, 26, 19, 30, 28, 17, 59, 32, 30, 52, 29, 31, 17,
    42, 41, 18, 16, 17, 93, 8, 25, 33, 17, 29, 40, 47, 28, 23, 33, 13, 30, 37, 25,
    42, 23, 10, 54, 25, 5, 31, 30, 10, 56, 12, 52, 48, 26, 51, 113, 20, 60, 13, 29,
    20, 55, 15, 14, 16, 40, 11, 14, 23, 26, 26, 15, 60, 71, 15, 18, 10, 21, 11, 51,
    28, 12, 23, 32, 17, 16, 26, 25, 48, 35, 38, 49, 63, 12, 7, 12, 32, 55, 19, 9,
    15, 75, 17, 13, 6, 25, 12, 5, 22, 34, 47, 25, 41, 37, 6, 26, 17, 106, 5, 25,
    18, 34, 37, 31, 17, 16, 33, 31, 20, 48, 53, 29, 7, 15, 23, 33, 24, 23, 13, 25,
    42, 9, 8, 9, 7, 43, 31, 14, 32, 85, 19, 25, 16, 35, 65, 10, 15, 32, 11, 12,
    12, 27, 30, 31, 23, 21, 27, 33, 36, 16, 75, 14, 21, 21, 20, 32, 65, 15, 31, 18,
    11, 34, 15, 54, 12, 37, 16, 30, 18, 25, 59, 23, 16, 13, 12, 70, 24, 29, 18, 28,
    9, 14, 10, 63, 15, 22, 10, 15, 12, 34, 9, 17, 20, 22, 19, 23, 19, 15, 21, 9,
    5, 39, 28, 26, 21, 27, 22, 17, 53, 7, 82, 59, 4, 23, 53, 27, 55, 14, 44, 21,
    25, 23, 32, 29, 34, 42, 32, 40, 63, 14, 34, 15, 17, 20, 21, 6, 17, 26, 48, 26,
    40, 26, 26, 13, 42, 53, 21, 41, 12, 8, 20, 21, 16, 53, 33, 17, 22, 14, 30, 57,
    35, 17, 12, 33, 17, 9, 7, 48, 21, 28, 26, 30, 26, 14, 20, 12, 22, 21, 45, 15,
    40, 17, 38, 13, 26, 26, 23, 12, 3, 26, 15, 8, 19, 18, 34, 16, 26, 9, 43, 20,
    7, 19, 32, 19, 17, 7, 12, 8, 10, 38, 37, 67, 26, 12, 16, 50, 16, 21, 17, 7,
    20, 22, 9, 23, 4, 21, 34, 24, 19, 26, 44, 21, 27, 18, 12, 25, 33, 41, 8, 19,
    33, 19, 22, 59, 29, 57, 39, 10, 12, 13, 12, 8, 12, 23, 16, 32, 15, 45, 34, 29,
    9, 45, 32, 27, 23, 52, 27, 37, 19, 20, 27, 14, 16, 28, 33, 10, 36, 31, 24, 20,
    22, 8, 22, 22, 7, 17, 33, 34, 41, 11, 14, 8, 8, 40, 23, 32, 66, 39, 20, 31,
    37, 24, 21, 18, 18, 17, 31, 50, 9, 61, 22, 58, 13, 16, 14, 4, 29, 34, 13, 22,
    8, 22, 77, 18, 59, 25, 12, 19, 16, 19, 20, 26, 45, 14, 42, 28, 16, 31, 25, 9,
    10, 28, 33, 36, 31, 34, 18, 37, 20, 28, 7, 24, 34, 21, 21, 15, 11, 21, 111, 16,
    8, 16, 27, 38, 15, 10, 31, 3, 15, 81, 6, 46, 17, 6, 7, 8, 51, 17, 36, 14,
    21, 39, 23, 12, 50, 36, 22, 23, 25, 47, 19, 33, 44, 10, 16, 47, 17, 11, 16, 9,
    43, 12, 22, 11, 22, 15, 22, 41, 10, 51, 18, 28, 13, 8, 18, 17, 14, 12, 19, 11,
    6, 17, 13, 23, 63, 18, 24, 16, 15, 32, 48, 25, 15, 27, 17, 22, 42, 29, 22, 6,
    7, 22, 30, 17, 15, 39, 17, 33, 43, 13, 65, 8, 10, 14, 33, 7, 8, 12, 51, 21,
    57, 46, 49, 17, 6, 31, 12, 24, 17, 16, 20, 9, 14, 16, 47, 27, 111, 6, 60, 33,
    17, 20, 23, 12, 36, 20, 94, 21, 11, 9, 19, 9, 39, 24, 22, 50, 51, 18, 15, 45,
    21, 22, 41, 16, 44, 12, 13, 20, 11, 49, 63, 17, 7, 33, 12, 13, 48, 30, 10, 74,
    17, 12, 38, 10, 16, 17, 20, 19, 6, 21, 39, 69, 35, 24, 17, 29, 9, 100, 6, 59,
    25, 29, 7, 26, 37, 11, 18, 29, 53, 36, 9, 17, 14, 26, 29, 16, 36, 15, 20, 16,
    46, 19, 25, 11, 57, 21, 5, 28, 36, 6, 29, 51, 12, 39, 8, 26, 12, 17, 16, 16,
    17, 18, 3, 13, 26, 22, 43, 19, 12, 30, 19, 56, 15, 7, 36, 43, 7, 12, 16, 48,
    34, 93, 22, 29, 12, 25, 14, 26, 58, 18, 29, 20, 10, 12, 18, 6, 13, 38, 21, 18,
    6, 41, 10, 29, 15, 17, 21, 30, 9, 8, 57, 15, 36, 42, 98, 16, 8, 15, 16, 9,
    20, 25, 19, 22, 29, 73, 20, 16, 24, 24, 25, 27, 16, 23, 52, 21, 19, 28, 9, 26,
    30, 23, 17, 55, 46, 10, 9, 13, 22, 18, 33, 15, 17, 55, 18, 36, 37, 12, 9, 38,
    28, 18, 15, 55, 17, 35, 29, 54, 8, 69, 14, 10, 19, 8, 23, 43, 37, 14, 15, 47,
    10, 29, 43, 44, 11, 66, 39, 13, 24, 16, 25, 13, 12, 30, 7, 31, 34, 26, 39, 90,
    15, 21, 77, 38, 23, 13, 25, 24, 31, 72, 25, 16, 17, 33, 6, 49, 14, 25, 9, 50,
    16, 34, 20, 5, 36, 15, 11, 58, 12, 24, 12, 34, 21, 18, 8, 28, 72, 14, 6, 27,
    26, 45, 17, 12, 9, 14, 7, 26, 18, 45, 49, 9, 6, 13, 54, 18, 21, 12, 22, 6,
    102, 36, 15, 28, 15, 23, 11, 7, 57, 31, 97, 20, 22, 22, 15, 12, 13, 33, 52, 35,
    31, 17, 15, 35, 9, 30, 21, 20, 23, 31, 88, 18, 30, 32, 42, 23, 44, 13, 55, 17,
    25, 24, 30, 51, 26, 21, 28, 60, 12, 20, 20, 19, 18, 47, 24, 27, 9, 10, 30, 15,
    48, 29, 59, 26, 20, 20, 42, 65, 10, 23, 38, 19, 11, 19, 56, 10, 37, 19, 52, 42,
    22, 25, 20, 15, 14, 12, 12, 12, 16, 19, 11, 8, 25, 16, 7, 15, 12, 43, 6, 21,
    15, 17, 37, 18, 16, 23, 41, 15, 22, 11, 18, 30, 21, 37, 11, 42, 106, 32, 17, 34,
    21, 16, 35, 23, 20, 26, 21, 24, 15, 31, 65, 38, 19, 41, 56, 48, 29, 20, 17, 52,
    42, 24, 24, 6, 11, 52, 10, 45, 16, 11, 16, 41, 40, 10, 23, 20, 15, 29, 43, 23,
    24, 10, 14, 27, 36, 10, 34, 53, 10, 18, 23, 40, 25, 31, 14, 34, 6, 32, 50, 38,
    32, 24, 32, 12, 17, 26, 41, 31, 9, 9, 45, 22, 52, 9, 41, 13, 13, 77, 26, 49,
    72, 34, 14, 20, 9, 27, 15, 16, 33, 17, 16, 65, 12, 14, 24, 21, 10, 50, 15, 30,
    28, 16, 29, 16, 10, 23, 28, 35, 22, 53, 26, 30, 6, 12, 18, 29, 10, 53, 15, 9,
    47, 6, 27, 14, 13, 38, 35, 21, 45, 8, 7, 15, 12, 20, 45, 8, 15, 29, 27, 52,
    12, 48, 10, 12, 10, 15, 23, 53, 22, 23, 36, 17, 25, 41, 42, 29, 23, 27, 29, 37,
    27, 33, 50, 22, 74, 48, 22, 16, 21, 85, 77, 23, 37, 43, 84, 19, 43, 13, 6, 10,
    11, 21, 37, 35, 18, 35, 43, 18, 18, 30, 19, 58, 20, 11, 30, 44, 15, 44, 10, 63,
    21, 26, 35, 72, 16, 15, 25, 8, 53, 9, 26, 19, 9, 59, 31, 21, 33, 14, 38, 58,
    32, 58, 33, 14, 22, 17, 11, 38, 25, 32, 12, 12, 28, 13, 20, 32, 16, 22, 17, 21,
    6, 21, 49, 17, 22, 79, 24, 34, 37, 42, 14, 27, 28, 28, 32, 41, 52, 17, 33, 47,
    38, 34, 45, 19, 16, 19, 36, 20, 40, 21, 16, 37, 31, 35, 12, 47, 16, 29, 29, 23,
    40, 27, 70, 24, 57, 21, 21, 24, 33, 12, 17, 16, 14, 17, 67, 9, 18, 43, 61, 15,
    8, 32, 24, 22, 6, 26, 9, 9, 32, 26, 61, 29, 13, 16, 43, 13, 19, 20, 27, 22,
    20, 9, 18, 16, 43, 22, 19, 22, 6, 28, 23, 20, 30, 24, 41, 26, 31, 49, 23, 19,
    14, 22, 16, 11, 29, 14, 63, 28, 16, 30, 15, 25, 19, 37, 58, 9, 27, 15, 22, 76,
    25, 15, 7, 18, 35, 18, 20, 18, 15, 13, 51, 23, 47, 20, 39, 36, 13, 29, 13, 26,
    61, 7, 51, 28, 31, 64, 13, 19, 18, 14, 17, 44, 9, 9, 16, 60, 77, 46, 55, 43,
    23, 11, 18, 36, 17, 30, 66, 16, 19, 20, 19, 32, 25, 47, 21, 23, 32, 7, 15, 34,
    34, 20, 10, 53, 30, 15, 21, 51, 12, 46, 15, 40, 62, 15, 36, 42, 10, 21, 24, 21,
    13, 40, 34, 91, 21, 32, 11, 17, 29, 13, 17, 11, 52, 66, 24, 12, 17, 11, 36, 43,
    11, 34, 37, 32, 9, 19, 17, 124, 24, 5, 17, 17, 33, 40, 33, 40, 13, 5, 12, 38,
    12, 19, 49, 21, 20, 9, 11, 13, 17, 10, 42, 23, 31, 11, 42, 66, 40, 20, 35, 20,
    21, 62, 20, 36, 28, 19, 10, 3, 15, 9, 16, 51, 11, 17, 98, 25, 10, 26, 34, 6,
    13, 17, 32, 11, 31, 31, 9, 27, 106, 18, 11, 17, 32, 24, 20, 90, 13, 24, 39, 21,
    16, 8, 16, 11, 48, 22, 37, 20, 21, 27, 11, 48, 7, 40, 16, 11, 59, 6, 25, 19,
    28, 43, 87, 5, 50, 18, 53, 64, 24, 75, 52, 50, 20, 19, 29, 25, 40, 28, 13, 38,
    25, 40, 26, 23, 22, 17, 15, 13, 10, 10, 15, 18, 24, 13, 20, 66, 49, 49, 24, 35,
    41, 15, 16, 9, 21, 15, 8, 26, 40, 36, 23, 24, 17, 38, 25, 18, 19, 45, 32, 17,
    9, 32, 12, 46, 9, 16, 23, 52, 16, 13, 12, 13, 31, 14, 29, 11, 39, 12, 20, 26,
    21, 22, 21, 29, 26, 15, 37, 93, 47, 23, 35, 29, 15, 19, 11, 14, 12, 19, 7, 25,
    10, 27, 32, 17, 18, 19, 7, 31, 5, 11, 10, 9, 30, 23, 6, 21, 37, 55, 26, 23,
    11, 14, 13, 66, 9, 25, 34, 25, 18, 30, 14, 9, 95, 14, 22, 51, 35, 18, 23, 16,
    25, 12, 57, 27, 25, 17, 11, 20, 30, 18, 13, 10, 16, 13, 54, 15, 44, 20, 13, 8,
    45, 16, 12, 39, 30, 92, 33, 31, 15, 6, 12, 43, 47, 16, 11, 21, 23, 44, 7, 72,
    24, 10, 26, 73, 20, 43, 14, 18, 23, 37, 6, 9, 7, 34, 14, 41, 43, 8, 14, 10,
    27, 19, 36, 19, 21, 29, 13, 24, 14, 29, 53, 16, 96, 24, 9, 34, 16, 46, 7, 11,
    24, 22, 13, 11, 23, 28, 30, 21, 70, 31, 25, 19, 22, 8, 13, 65, 29, 44, 48, 21,
    41, 12, 23, 22, 16, 50, 8, 31, 29, 7, 40, 17, 23, 42, 24, 57, 19, 15, 8, 102,
    6, 44, 10, 14, 34, 7, 30, 16, 9, 39, 31, 10, 23, 27, 27, 9, 7, 41, 26, 20,
    38, 40, 29, 9, 13, 20, 11, 12, 40, 33, 18, 24, 17, 16, 14, 35, 36, 17, 23, 15,
    24, 30, 38, 9, 29, 18, 41, 14, 19, 6, 17, 16, 10, 18, 17, 30, 23, 21, 13, 9,
    36, 10, 22, 11, 49, 10, 10, 17, 42, 44, 17, 8, 45, 53, 11, 6, 52, 40, 16, 26,
    20, 40, 47, 48, 22, 9, 27, 42, 24, 62, 15, 29, 13, 23, 13, 35, 37, 45, 23, 32,
    33, 12, 16, 21, 13, 43, 33, 32, 40, 31, 45, 37, 16, 34, 71, 65, 23, 29, 18, 42,
    23, 10, 28, 13, 21, 35, 24, 14, 16, 10, 17, 28, 109, 21, 45, 20, 19, 23, 53, 42,
    9, 28, 15, 19, 22, 21, 56, 15, 14, 13, 56, 21, 26, 8, 27, 21, 12, 24, 12, 28,
    47, 28, 10, 41, 20, 16, 19, 37, 63, 26, 28, 29, 60, 23, 40, 23, 19, 6, 10, 33,
    40, 22, 12, 42, 45, 11, 45, 29, 63, 30, 44, 12, 31, 46, 11, 30, 52, 40, 18, 12,
    21, 17, 55, 25, 17, 34, 53, 13, 14, 18, 61, 18, 8, 19, 20, 14, 17, 56, 21, 17,
    19, 35, 4, 13, 26, 22, 14, 21, 50, 47, 24, 9, 22, 46, 11, 14, 15, 13, 16, 16,
    30, 16, 60, 17, 31, 14, 11, 14, 10, 11, 24, 22, 28, 18, 20, 59, 28, 27, 26, 34,
    19, 9, 21, 10, 17, 17, 67, 21, 51, 29, 46, 14, 24, 24, 23, 17, 9, 11, 12, 36,
    14, 23, 12, 6, 39, 8, 63, 25, 14, 30, 38, 41, 7, 22, 24, 11, 22, 31, 9, 13,
    10, 124, 9, 22, 19, 19, 14, 38, 40, 8, 15, 19, 33, 28, 27, 40, 40, 25, 17, 8,
    28, 31, 21, 19, 25, 21, 23, 67, 23, 15, 19, 12, 18, 18, 20, 23, 20, 24, 45, 13,
    44, 28, 18, 28, 18, 11, 17, 24, 8, 12, 56, 17, 14, 13, 24, 30, 64, 17, 25, 12,
    11, 39, 16, 31, 28, 13, 83, 8, 16, 74, 22, 26, 7, 21, 6, 16, 27, 6, 24, 17,
    36, 22, 12, 7, 19, 29, 39, 95, 16, 34, 12, 10, 18, 13, 16, 17, 49, 22, 27, 25,
    18, 26, 15, 13, 18, 26, 31, 14, 42, 42, 13, 55, 5, 37, 12, 29, 17, 48, 26, 24,
    21, 20, 14, 25, 19, 38, 15, 29, 51, 14, 44, 46, 16, 22, 24, 28, 11, 19, 39, 27,
    8, 26, 10, 25, 15, 26, 15, 17, 8, 29, 16, 10, 17, 20, 22, 39, 22, 67, 13, 12,
    18, 6, 11, 13, 38, 18, 18, 22, 11, 9, 10, 63, 26, 15, 19, 6, 8, 22, 71, 36,
    14, 25, 57, 50, 22, 15, 11, 8, 39, 12, 8, 14, 24, 10, 10, 14, 21, 14, 31, 52,
    47, 5, 24, 13, 11, 28, 14, 6, 8, 5, 39, 19, 43, 24, 16, 19, 21, 18, 7, 43,
    13, 17, 11, 40, 17, 22, 22, 18, 14, 11, 99, 37, 18, 21, 48, 25, 23, 18, 26, 18,
    27, 18, 42, 20, 34, 18, 13, 6, 31, 15, 14, 36, 26, 19, 32, 30, 39, 66, 14, 20,
    17, 48, 17, 35, 38, 10, 61, 11, 13, 29, 48, 21, 14, 16, 6, 18, 8, 22, 50, 28,
    16, 17, 14, 17, 11, 6, 9, 24, 46, 40, 43, 31, 32, 24, 11, 96, 9, 28, 19, 29,
    19, 46, 22, 3, 14, 7, 21, 24, 20, 24, 23, 45, 31, 50, 34, 17, 6, 29, 27, 17,
    6, 23, 10, 5, 18, 23, 10, 21, 6, 37, 25, 23, 9, 61, 21, 41, 9, 22, 10, 17,
    33, 20, 33, 15, 15, 36, 25, 27, 33, 24, 9, 19, 36, 19, 30, 14, 49, 14, 21, 15,
    51, 31, 10, 24, 23, 22, 26, 67, 52, 30, 16, 13, 17, 27, 30, 40, 3, 27, 14, 13,
    13, 18, 17, 18, 12, 13, 15, 39, 10, 16, 45, 47, 10, 34, 13, 28, 21, 30, 42, 34,
    85, 15, 24, 46, 116, 33, 22, 36, 35, 11, 4, 12, 16, 21, 64, 15, 14, 5, 35, 39,
    19, 22, 18, 12, 28, 14, 27, 20, 78, 12, 20, 50, 28, 17, 28, 7, 26, 19, 29, 21,
    59, 9, 164, 55, 23, 19, 42, 55, 20, 14, 59, 32, 15, 27, 25, 6, 16, 11, 21, 13,
    13, 46, 11, 6, 52, 28, 16, 15, 4, 35, 33, 21, 15, 15, 14, 45, 19, 10, 47, 79,
    26, 40, 33, 44, 22, 17, 19, 13, 24, 8, 19, 18, 32, 65, 34, 27, 36, 25, 65, 24,
    14, 23, 69, 31, 29, 22, 24, 63, 17, 21, 25, 30, 29, 16, 13, 42, 24, 30, 26, 24,
    45, 52, 18, 25, 50, 23, 38, 36, 22, 19, 33, 16, 22, 10, 15, 6, 8, 12, 33, 19,
    55, 11, 13, 12, 20, 10, 7, 28, 35, 38, 25, 41, 16, 42, 4, 29, 17, 39, 29, 15,
    12, 10, 15, 17, 23, 14, 20, 6, 10, 23, 31, 36, 34, 14, 73, 15, 10, 18, 11, 10,
    14, 31, 19, 25, 40, 95, 9, 46, 24, 29, 48, 41, 8, 10, 29, 30, 10, 23, 15, 12,
    24, 13, 15, 49, 42, 34, 16, 16, 38, 18, 29, 27, 31, 51, 29, 25, 15, 22, 50, 19,
    23, 42, 36, 12, 8, 27, 16, 13, 12, 54, 57, 28, 17, 23, 39, 37, 56, 19, 24, 28,
    14, 27, 14, 22, 33, 35, 11, 40, 19, 26, 20, 33, 19, 16, 15, 20, 30, 10, 81, 27,
    41, 15, 27, 11, 20, 18, 17, 51, 18, 43, 19, 17, 19, 19, 28, 43, 29, 24, 43, 25,
    28, 21, 32, 10, 67, 20, 40, 28, 14, 34, 30, 20, 14, 32, 15, 37, 36, 30, 67, 20,
    16, 63, 8, 71, 9, 15, 32, 9, 15, 20, 55, 22, 42, 25, 22, 16, 16, 42, 10, 75,
    40, 13, 24, 31, 27, 17, 12, 15, 40, 15, 36, 46, 33, 19, 28, 15, 22, 10, 51, 30,
    48, 42, 22, 58, 20, 22, 19, 22, 12, 24, 13, 39, 27, 28, 16, 17, 12, 14, 22, 11,
    17, 22, 26, 9, 41, 12, 57, 15, 23, 35, 45, 14, 21, 10, 14, 8, 14, 26, 16, 27,
    12, 21, 68, 49, 7, 18, 23, 48, 16, 16, 64, 25, 24, 13, 36, 14, 42, 18, 37, 13,
    19, 20, 55, 16, 11, 13, 29, 24, 22, 16, 15, 14, 25, 29, 30, 26, 7, 17, 12, 19,
    22, 32, 26, 38, 27, 6, 31, 80, 39, 100, 38, 8, 20, 22, 112, 24, 16, 24, 10, 77,
    12, 19, 26, 16, 24, 10, 35, 8, 13, 28, 88, 21, 10, 27, 15, 27, 26, 47, 69, 39,
    28, 13, 23, 22, 19, 21, 17, 45, 41, 26, 31, 15, 11, 71, 27, 18, 8, 14, 19, 13,
    19, 21, 15, 22, 10, 21, 16, 17, 5, 42, 56, 17, 11, 8, 19, 17, 20, 19, 16, 7,
    45, 32, 14, 18, 18, 13, 17, 20, 26, 22, 21, 15, 25, 16, 17, 53, 11, 53, 28, 25,
    27, 13, 34, 19, 30, 14, 18, 24, 93, 10, 31, 19, 20, 21, 45, 30, 68, 71, 26, 16,
    14, 19, 9, 18, 43, 24, 23, 51, 36, 29, 44, 10, 17, 10, 18, 22, 9, 11, 21, 37,
    55, 25, 24, 33, 63, 17, 48, 8, 30, 46, 63, 27, 37, 30, 41, 18, 13, 13, 8, 18,
    16, 53, 10, 9, 44, 38, 25, 49, 30, 15, 37, 7, 26, 77, 14, 14, 28, 47, 34, 21,
    15, 43, 9, 29, 35, 19, 22, 38, 32, 29, 21, 19, 40, 22, 63, 24, 20, 10, 9, 5,
    14, 26, 16, 18, 136, 11, 29, 34, 23, 22, 34, 25, 20, 40, 10, 9, 20, 11, 36, 21,
    47, 33, 31, 23, 10, 15, 59, 5, 13, 21, 59, 77, 26, 47, 16, 10, 108, 15, 8, 22,
    21, 16, 73, 22, 21, 12, 7, 13, 41, 6, 6, 11, 16, 18, 16, 47, 57, 22, 43, 21,
    16, 30, 25, 14, 36, 31, 23, 10, 32, 8, 23, 15, 40, 35, 12, 16, 42, 71, 39, 15,
    9, 11, 78, 60, 55, 51, 22, 62, 25, 35, 34, 20, 25, 20, 15, 46, 12, 18, 50, 26,
    20, 27, 131, 9, 37, 69, 40, 30, 50, 13, 36, 55, 51, 14, 22, 32, 43, 42, 13, 11,
    11, 62, 39, 24, 14, 14, 29, 21, 9, 24, 11, 17, 14, 15, 22, 43, 13, 29, 23, 33,
    21, 33, 17, 32, 24, 30, 49, 27, 34, 16, 15, 69, 19, 7, 14, 30, 52, 16, 27, 25,
    15, 26, 24, 12, 16, 5, 23, 17, 8, 25, 15, 18, 20, 46, 17, 44, 108, 47, 69, 24,
    18, 7, 24, 36, 49, 15, 24, 8, 13, 27, 14, 54, 9, 52, 97, 14, 49, 21, 11, 8,
    79, 21, 11, 30, 14, 14, 14, 9, 21, 9, 62, 23, 12, 36, 25, 19, 12, 27, 35, 19,
    27, 13, 21, 24, 70, 20, 35, 18, 48, 8, 14, 28, 53, 12, 14, 19, 19, 15, 16, 33,
    17, 27, 78, 101, 13, 40, 18, 20, 39, 9, 83, 29, 20, 10, 42, 26, 8, 75, 27, 17,
    43, 7, 12, 20, 29, 10, 16, 51, 19, 24, 45, 18, 15, 26, 13, 15, 21, 38, 71, 13,
    18, 44, 13, 39, 30, 53, 31, 12, 34, 55, 19, 84, 12, 24, 29, 22, 59, 22, 47, 17,
    11, 36, 50, 53, 51, 28, 22, 26, 6, 19, 14, 47, 30, 15, 16, 14, 12, 19, 35, 24,
    16, 22, 13, 13, 25, 16, 13, 16, 59, 24, 30, 22, 17, 6, 33, 19, 47, 22, 29, 22,
    11, 32, 21, 23, 19, 74, 17, 15, 44, 34, 52, 11, 36, 14, 21, 10, 9, 42, 34, 28,
    78, 26, 17, 36, 26, 14, 10, 8, 81, 19, 28, 13, 25, 38, 38, 19, 24, 11, 14, 17,
    13, 52, 14, 44, 43, 58, 29, 8, 34, 53, 17, 24, 39, 32, 13, 39, 36, 22, 54, 18,
    35, 23, 31, 64, 27, 22, 10, 16, 14, 31, 21, 38, 31, 5, 13, 21, 42, 13, 25, 66,
    11, 20, 42, 30, 42, 27, 30, 41, 35, 17, 19, 24, 38, 34, 33, 33, 16, 27, 46, 20,
    77, 19, 92, 66, 23, 8, 13, 56, 17, 11, 19, 70, 16, 13, 17, 24, 27, 14, 16, 45,
    45, 14, 14, 50, 14, 5, 26, 14, 21, 10, 48, 6, 9, 17, 27, 17, 94, 11, 28, 49,
    10, 16, 7, 45, 9, 25, 28, 19, 12, 15, 43, 29, 22, 24, 26, 39, 15, 39, 27, 30,
    17, 33, 24, 72, 9, 19, 10, 19, 19, 11, 43, 9, 27, 15, 14, 19, 42, 44, 57, 17,
    57, 31, 22, 44, 33, 21, 16, 47, 11, 28, 16, 23, 107, 41, 37, 31, 25, 20, 19, 58,
    16, 8, 12, 36, 7, 21, 28, 10, 12, 13, 103, 42, 13, 11, 26, 21, 25, 12, 14, 26,
    25, 12, 25, 38, 32, 16, 35, 15, 14, 12, 35, 31, 8, 20, 17, 15, 19, 10, 12, 83,
    17, 8, 38, 10, 42, 16, 27, 24, 38, 88, 24, 20, 14, 30, 9, 21, 30, 49, 13, 19,
    28, 37, 29, 31, 20, 18, 46, 18, 27, 28, 34, 55, 18, 11, 21, 69, 39, 18, 14, 5,
    7, 21, 16, 33, 19, 15, 22, 14, 32, 59, 16, 41, 9, 23, 9, 17, 44, 24, 41, 9,
    18, 42, 10, 26, 19, 11, 14, 8, 27, 8, 60, 28, 24, 39, 44, 27, 42, 14, 12, 35,
    23, 9, 13, 43, 26, 16, 22, 50, 60, 37, 17, 28, 27, 28, 31, 25, 26, 21, 61, 34,
    20, 13, 61, 7, 28, 14, 22, 34, 21, 16, 18, 29, 9, 23, 5, 28, 8, 11, 25, 32,
    28, 33, 18, 77, 13, 14, 42, 26, 43, 19, 23, 32, 33, 18, 10, 24, 54, 28, 50, 12,
    59, 43, 85, 10, 40, 44, 30, 29, 23, 20, 22, 31, 12, 13, 13, 8, 23, 49, 38, 34,
    18, 23, 11, 38, 22, 17, 19, 58, 15, 14, 50, 26, 7, 9, 20, 18, 16, 18, 25, 13,
    14, 39, 20, 77, 34, 20, 34, 27, 16, 37, 16, 25, 32, 13, 15, 75, 44, 9, 24, 17,
    15, 7, 11, 36, 12, 54, 38, 53, 16, 12, 13, 18, 34, 21, 34, 24, 22, 79, 31, 22,
    13, 16, 10, 25, 8, 16, 15, 21, 34, 13, 13, 44, 13, 29, 45, 44, 53, 9, 32, 31,
    28, 20, 16, 20, 11, 35, 19, 12, 122, 15, 34, 21, 31, 26, 21, 9, 29, 27, 27, 28,
    10, 17, 22, 47, 27, 88, 13, 32, 49, 18, 21, 64, 18, 11, 31, 17, 9, 31, 7, 31,
    7, 40, 8, 46, 9, 50, 38, 23, 62, 12, 19, 26, 61, 32, 17, 35, 88, 29, 26, 22,
    27, 32, 18, 29, 10, 21, 23, 15, 28, 34, 40, 51, 22, 27, 18, 15, 40, 44, 23, 25,
    8, 22, 21, 18, 22, 15, 7, 18, 28, 58, 32, 35, 26, 51, 12, 34, 10, 17, 11, 27,
    16, 18, 45, 27, 30, 27, 13, 36, 21, 19, 30, 29, 28, 15, 17, 19, 25, 26, 14, 15,
    6, 30, 30, 25, 60, 18, 66, 34, 15, 23, 13, 25, 33, 47, 18, 17, 17, 36, 32, 36,
    35, 29, 62, 15, 10, 11, 52, 23, 32, 23, 25, 25, 26, 10, 39, 23, 16, 9, 35, 42,
    52, 17, 7, 13, 58, 16, 29, 45, 20, 27, 21, 14, 42, 18, 40, 39, 20, 18, 23, 24,
    18, 24, 4, 24, 18, 26, 44, 12, 25, 8, 7, 16, 38, 10, 18, 16, 26, 13, 29, 12,
    61, 22, 29, 28, 23, 17, 12, 31, 37, 24, 22, 14, 18, 21, 55, 18, 15, 4, 17, 12,
    14, 49, 24, 20, 15, 21, 7, 21, 63, 6, 28, 21, 20, 29, 24, 11, 32, 44, 22, 33,
    17, 36, 17, 34, 7, 33, 24, 24, 22, 21, 26, 12, 7, 54, 31, 63, 34, 21, 25, 13,
    9, 20, 39, 45, 43, 24, 38, 20, 38, 29, 22, 7, 27, 12, 18, 45, 52, 12, 20, 22,
    21, 6, 52, 43, 7, 19, 31, 15, 27, 20, 12, 11, 16, 22, 27, 19, 42, 53, 29, 48,
    17, 29, 13, 43, 12, 21, 14, 16, 48, 28, 27, 17, 14, 38, 40, 55, 38, 11, 16, 27,
    23, 16, 84, 22, 14, 47, 8, 18, 36, 17, 26, 15, 12, 31, 21, 21, 49, 59, 25, 10,
    15, 35, 21, 14, 13, 36, 31, 10, 31, 19, 18, 47, 19, 8, 21, 5, 11, 20, 19, 23,
    72, 24, 10, 24, 87, 17, 19, 15, 22, 32, 11, 37, 24, 9, 25, 13, 38, 52, 12, 37,
    29, 11, 34, 11, 21, 25, 21, 86, 28, 4, 31, 33, 23, 32, 34, 38, 54, 19, 19, 17,
    25, 30, 53, 27, 22, 39, 11, 43, 38, 69, 14, 11, 65, 8, 11, 12, 35, 20, 26, 46,
    17, 17, 96, 33, 11, 30, 24, 39, 10, 28, 9, 14, 52, 22, 19, 49, 22, 22, 37, 21,
    42, 12, 19, 126, 15, 30, 35, 11, 29, 34, 23, 70, 28, 74, 23, 21, 30, 8, 17, 19,
    34, 14, 33, 8, 30, 49, 22, 8, 15, 44, 12, 14, 40, 21, 18, 105, 28, 9, 19, 16,
    24, 16, 22, 40, 18, 22, 46, 16, 15, 42, 57, 37, 49, 28, 19, 28, 28, 58, 24, 42,
    18, 9, 10, 11, 15, 31, 8, 25, 28, 22, 15, 31, 16, 34, 12, 51, 28, 29, 24, 11,
    36, 12, 36, 16, 45, 18, 25, 20, 20, 30, 107, 14, 20, 15, 19, 6, 15, 19, 10, 23,
    20, 9, 34, 29, 8, 21, 127, 20, 24, 26, 47, 40, 24, 26, 15, 80, 30, 8, 23, 22,
    21, 13, 27, 26, 25, 39, 19, 16, 33, 47, 102, 57, 40, 21, 21, 35, 24, 34, 22, 11,
    31, 25, 32, 66, 18, 14, 49, 30, 37, 22, 31, 24, 22, 34, 28, 15, 21, 22, 17, 41,
    19, 21, 28, 13, 37, 21, 9, 6, 30, 17, 34, 18, 40, 20, 40, 13, 39, 20, 62, 27,
    41, 15, 29, 16, 6, 27, 9, 18, 54, 38, 27, 65, 31, 13, 15, 30, 9, 23, 32, 32,
    10, 39, 51, 30, 32, 86, 41, 13, 26, 7, 24, 13, 19, 9, 13, 40, 47, 48, 14, 39,
    13, 15, 16, 65, 41, 37, 21, 38, 39, 9, 41, 83, 7, 4, 23, 12, 14, 7, 89, 40,
    23, 14, 34, 30, 28, 22, 65, 21, 13, 88, 36, 31, 37, 18, 16, 10, 50, 47, 23, 7,
    7, 13, 22, 12, 33, 36, 45, 25, 43, 23, 26, 11, 40, 33, 41, 16, 25, 6, 23, 18,
    29, 9, 222, 33, 57, 12, 35, 24, 27, 27, 27, 67, 40, 18, 14, 18, 14, 20, 34, 54,
    34, 48, 12, 13, 19, 19, 55, 75, 34, 25, 21, 15, 14, 11, 21, 24, 14, 21, 77, 37,
    21, 19, 29, 29, 27, 13, 22, 32, 38, 15, 14, 19, 10, 29, 18, 38, 153, 7, 31, 20,
    22, 75, 11, 26, 31, 12, 9, 8, 17, 15, 33, 39, 15, 26, 9, 54, 17, 32, 47, 14,
    68, 11, 73, 22, 13, 7, 11, 23, 31, 75, 34, 10, 34, 7, 23, 29, 11, 33, 22, 50,
    15, 27, 16, 50, 35, 17, 18, 10, 13, 17, 44, 35, 30, 21, 24, 15, 13, 13, 57, 16,
    26, 8, 8, 13, 30, 17, 49, 54, 50, 38, 9, 25, 21, 15, 49, 18, 43, 29, 15, 64,
    37, 16, 11, 37, 9, 25, 18, 17, 57, 14, 26, 25, 11, 22, 16, 34, 15, 28, 31, 7,
    22, 47, 27, 18, 23, 11, 36, 39, 21, 48, 21, 74, 49, 22, 10, 23, 14, 31, 27, 13,
    25, 12, 42, 8, 43, 41, 18, 46, 59, 37, 105, 21, 44, 18, 20, 12, 24, 15, 13, 72,
    15, 83, 12, 25, 20, 28, 13, 112, 18, 20, 10, 30, 7, 7, 8, 14, 58, 39, 67, 17,
    8, 22, 22, 17, 12, 47, 17, 30, 16, 23, 30, 40, 9, 16, 19, 16, 22, 68, 13, 24,
    15, 22, 21, 6, 16, 6, 12, 32, 19, 34, 15, 32, 18, 11, 17, 79, 29, 26, 39, 59,
    12, 13, 20, 17, 45, 15, 24, 6, 43, 18, 34, 9, 28, 7, 9, 11, 18, 24, 5, 39,
    65, 19, 22, 17, 37, 15, 26, 61, 27, 10, 39, 11, 27, 42, 24, 43, 47, 30, 20, 22,
    17, 20, 16, 17, 21, 9, 10, 24, 35, 16, 10, 12, 9, 25, 31, 18, 38, 18, 17, 61,
    17, 48, 32, 60, 18, 12, 30, 20, 6, 11, 15, 34, 19, 4, 7, 52, 4, 34, 45, 8,
    13, 21, 22, 30, 24, 11, 17, 30, 57, 55, 34, 17, 9, 16, 44, 17, 17, 16, 9, 20,
    10, 26, 21, 11, 39, 13, 18, 25, 22, 15, 34, 6, 47, 54, 45, 23, 19, 22, 21, 18,
    36, 49, 27, 18, 31, 22, 19, 76, 37, 31, 16, 7, 36, 38, 23, 15, 35, 19, 60, 9,
    24, 39, 14, 19, 23, 28, 27, 45, 12, 8, 18, 15, 13, 104, 35, 11, 21, 29, 45, 51,
    32, 19, 24, 12, 30, 11, 13, 12, 61, 26, 34, 52, 14, 15, 31, 49, 24, 15, 21, 17,
    12, 59, 9, 48, 49, 6, 32, 23, 21, 18, 17, 13, 22, 22, 17, 20, 28, 18, 27, 25,
    20, 24, 15, 13, 70, 17, 26, 44, 23, 45, 16, 15, 14, 22, 32, 28, 42, 37, 20, 4,
    22, 13, 25, 26, 3, 15, 46, 19, 22, 28, 38, 50, 56, 28, 9, 21, 62, 44, 74, 29,
    25, 30, 19, 56, 24, 17, 20, 13, 22, 9, 16, 23, 12, 27, 28, 28, 46, 24, 22, 54,
    21, 25, 18, 32, 44, 18, 90, 58, 64, 26, 17, 17, 38, 44, 18, 10, 25, 17, 13, 24,
    16, 19, 25, 33, 51, 25, 42, 16, 23, 15, 52, 76, 26, 9, 66, 15, 19, 17, 22, 24,
    7, 45, 14, 30, 45, 7, 17, 41, 28, 30, 20, 11, 22, 24, 18, 20, 40, 7, 8, 56,
    32, 40, 16, 31, 43, 51, 24, 28, 14, 9, 15, 19, 62, 82, 34, 49, 21, 29, 24, 16,
    30, 19, 48, 13, 16, 66, 33, 36, 14, 37, 31, 23, 27, 55, 20, 71, 26, 48, 27, 5,
    43, 25, 11, 7, 24, 13, 29, 8, 32, 24, 23, 14, 18, 26, 35, 43, 15, 61, 16, 15,
    43, 5, 31, 9, 13, 24, 53, 14, 11, 25, 9, 20, 20, 21, 44, 22, 62, 22, 15, 22,
    29, 28, 48, 15, 21, 15, 48, 32, 20, 18, 9, 6, 71, 30, 23, 22, 20, 52, 45, 14,
    5, 41, 21, 12, 17, 19, 18, 18, 70, 12, 14, 26, 27, 20, 42, 29, 9, 12, 23, 39,
    18, 15, 4, 18, 18, 30, 14, 17, 43, 60, 16, 33, 13, 20, 17, 14, 8, 15, 37, 31,
    50, 18, 8, 12, 19, 26, 42, 13, 11, 29, 36, 55, 21, 7, 28, 11, 22, 27, 21, 106,
    15, 17, 53, 11, 12, 21, 32, 15, 25, 21, 18, 6, 17, 21, 10, 28, 28, 19, 36, 16,
    23, 15, 36, 21, 32, 8, 15, 28, 52, 17, 68, 16, 8, 11, 58, 24, 23, 108, 32, 14,
    14, 8, 7, 22, 24, 8, 45, 19, 43, 25, 16, 43, 17, 72, 10, 18, 43, 14, 20, 41,
    23, 19, 16, 39, 14, 21, 23, 16, 10, 23, 20, 17, 32, 24, 17, 15, 23, 16, 95, 23,
    19, 41, 15, 18, 53, 54, 49, 34, 47, 34, 13, 38, 55, 31, 35, 19, 23, 32, 26, 22,
    27, 27, 23, 11, 9, 23, 21, 11, 22, 39, 52, 13, 18, 25, 27, 47, 25, 27, 5, 20,
    10, 42, 19, 23, 22, 21, 26, 11, 57, 12, 10, 7, 43, 31, 11, 95, 18, 17, 28, 33,
]
print(f"Ready: {len(population)} tickets, true mean {statistics.mean(population):.1f} minutes")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One test set is one sample

First, look at the whole population. Run the cell. You should see a true mean of 26.8 minutes and an SD of 17.8 minutes.

In [ ]:
true_mean = statistics.mean(population)
true_sd = statistics.pstdev(population)    # the whole population: divide by n
print("true mean", round(true_mean, 1), "minutes")
print("SD       ", round(true_sd, 1), "minutes")

The next cell defines `interval(values)`. It returns the sample mean, the standard error (SE) and the 95% interval, as the explorer does. Sections 2 and 3 explain each part. The multiplier depends on the sample size: 2.045 for 30 values, 1.972 for 200. The explorer uses the same numbers; `scipy.stats.t.ppf(0.975, n - 1)` gives them.

Then the cell draws one sample of 30 tickets, at random, with a fixed seed. Run it. You should see a sample mean of 24.0 minutes, an SE of 3.2 minutes and an interval of 17.5 to 30.5 minutes. It contains the true mean.

In [ ]:
MULTIPLIER = {5: 2.776, 12: 2.201, 30: 2.045, 50: 2.010, 200: 1.972}

def interval(values):
    """The mean, the SE and the 95% interval of one sample."""
    n = len(values)
    mean = statistics.mean(values)
    se = statistics.stdev(values) / math.sqrt(n)    # a sample: divide by n - 1
    half = MULTIPLIER[n] * se
    return mean, se, mean - half, mean + half

random.seed(5)
sample = random.choices(population, k=30)          # 30 tickets, picked at random
mean, se, low, high = interval(sample)
print(f"sample mean {mean:.1f}, SE {se:.1f}, 95% interval {low:.1f} to {high:.1f} minutes")
print("contains the true mean:", low <= true_mean <= high)

Now repeat this 100 times, as the explorer does.

> **Predict.** How many of 100 intervals from samples of 30 tickets contain the true mean? Write it down, then run the cell.

In [ ]:
def coverage(n, samples=100, seed=1):
    """Draw `samples` samples of n tickets; count the intervals that contain the true mean."""
    random.seed(seed)
    hits, widths, means = 0, [], []
    for _ in range(samples):
        mean, se, low, high = interval(random.choices(population, k=n))
        hits += low <= true_mean <= high
        widths.append(high - low)
        means.append(mean)
    return hits, statistics.mean(widths), means

hits, width, means = coverage(30)
print(f"{hits} of 100 intervals contain the true mean; mean width {width:.1f} minutes")

> **Check.** You should see `95 of 100` and a mean width of 12.4 minutes. Another seed gives another count, for example 90 or 96. About 5 of 100 intervals miss, and nothing in the sample shows which ones.

> **Your turn: change one thing.** In the cell above, change `coverage(30)` to `coverage(200)`, then to `coverage(5)`, and run it each time. With 200 tickets you should see `96 of 100` and a width of 4.9 minutes. With 5 tickets you should see `89 of 100` and a width of 39.8 minutes: very wide, and fewer than 95 hits, because the data has a long tail.

## 2. Standard error: the typical size of the chance error

The **standard error (SE)** is the typical distance between a sample mean and the true mean. For a mean, SE = SD ÷ √n. The next cell prints the lesson's table with the population SD of 17.8 minutes.

> **Predict.** From 50 to 200 tickets, n grows 4 times. How much smaller is the SE? Then run the cell.

In [ ]:
rows = [[n, math.sqrt(n), 17.8 / math.sqrt(n)] for n in [5, 30, 50, 200]]
table(["Tickets n", "√n", "SE (minutes)"], rows, digits=2)

> **Check.** You should see SE values of 7.96, 3.25, 2.52 and 1.26 minutes. The lesson writes 7.9, 3.2, 2.5 and 1.3: it divides the unrounded SD, 17.75 minutes, so 17.75 ÷ 2.24 = 7.94. Four times more tickets give half the SE: this is the **square-root rule**.

In the explorer, the SE is the SD of the sample means. The next cell checks this with 1,000 samples of each size. Run it. The two columns should be close.

In [ ]:
rows = []
for n in [30, 200]:
    _, _, means = coverage(n, samples=1000)
    rows.append([n, statistics.stdev(means), 17.8 / math.sqrt(n)])
table(["Tickets n", "SD of 1,000 sample means", "SD ÷ √n"], rows)

> **Check.** You should see 3.21 and 3.25 for 30 tickets, and 1.25 and 1.26 for 200 tickets. One sample is enough to estimate how much the means of many samples scatter.

## 3. Confidence interval: where 1.96 comes from

A 95% interval is the estimate ± 1.96 × SE. The 1.96 comes from the normal curve: 95% of it is within 1.96 SD of its centre. Python's `statistics.NormalDist` gives it. Run the cell. You should see `1.96` and `0.95`.

In [ ]:
z = statistics.NormalDist().inv_cdf(0.975)    # 2.5% is above it, 2.5% below -z
print(round(z, 2))
print(round(statistics.NormalDist().cdf(z) - statistics.NormalDist().cdf(-z), 2))

For small samples, software uses a slightly larger multiplier (the **t multiplier**): 2.20 for 12 values, 2.045 for 30. `MULTIPLIER` in section 1 has these values.

## 4. Worked example 1: the mean of 12 reply times

Priya calculates the 95% interval for the mean of the 12 Monday reply times. Here the 12 tickets are a sample, so the SD divides by n − 1 (`statistics.stdev`).

Run the cell. You should see the lesson's steps: SD 22.04, SE 6.36, half-width 14.0 and the interval **14.7 to 42.8 minutes**.

In [ ]:
monday = [12, 15, 18, 18, 20, 22, 24, 25, 27, 31, 38, 95]
n = len(monday)
mean = statistics.mean(monday)
sd = statistics.stdev(monday)
se = sd / math.sqrt(n)
half = MULTIPLIER[n] * se
table(["Step", "Value"], [
    ["Number of tickets, n", n],
    ["Mean (minutes)", float(mean)],
    ["SD, dividing by n - 1", sd],
    ["√n", math.sqrt(n)],
    ["SE = SD ÷ √n", se],
    ["Multiplier for 12 values", MULTIPLIER[n]],
    ["Half-width", half],
    ["95% interval, low", mean - half],
    ["95% interval, high", mean + half],
])
print()
print("divide by n:    ", round(statistics.pstdev(monday), 2))
print("divide by n - 1:", round(statistics.stdev(monday), 2))

> **Check.** The table shows the interval with two decimals: 14.75 to 42.75 minutes. With more decimals it is 14.748 to 42.752, so the lesson rounds it to 14.7 to 42.8. The last two lines show the two SDs from the lesson, 21.1 and 22.04: dividing by n − 1 makes the SD a little larger.

The `interval` function from section 1 gives the same result. Run the cell. You should see `14.75 to 42.75`.

In [ ]:
mean, se, low, high = interval(monday)
print(f"{low:.2f} to {high:.2f}")

## 5. Worked example 2: the accuracy of model B

An accuracy is the mean of many 1s (correct) and 0s (wrong). Model B is correct on 276 of 300 test tickets. The SE of a share p is √(p(1 − p) / n).

Run the cell. You should see the lesson's steps: SE 1.57 points, half-width 3.07 points and the interval **88.9% to 95.1%**.

In [ ]:
results_b = [1] * 276 + [0] * 24          # 1 = correct, 0 = wrong
n = len(results_b)
p = statistics.mean(results_b)
se = math.sqrt(p * (1 - p) / n)
half = 1.96 * se
table(["Step", "Value"], [
    ["p = 276 ÷ 300", p],
    ["p × (1 − p)", p * (1 - p)],
    ["÷ n", f"{p * (1 - p) / n:.6f}"],
    ["SE (points)", 100 * se],
    ["Half-width (points)", 100 * half],
    ["95% interval, low (%)", 100 * (p - half)],
    ["95% interval, high (%)", 100 * (p + half)],
], digits=4)

> **Check.** You should see p = 0.9200, p × (1 − p) = 0.0736, ÷ n = 0.000245, an SE of 1.5663 points, a half-width of 3.0700 points, and 88.9300% to 95.0700%. The ± is in **percentage points**, not "3.1% of 92%".

The formula √(p(1 − p)) is the SD of the 1s and 0s. So this is the same rule as SE = SD ÷ √n. Run the cell. You should see the same SE twice: `1.5663`.

In [ ]:
print(round(100 * math.sqrt(p * (1 - p) / n), 4))
print(round(100 * statistics.pstdev(results_b) / math.sqrt(n), 4))

## 6. The bootstrap: resample the test set

The **bootstrap** builds new test sets of 300 tickets: it picks tickets **from the 300**, at random, **with replacement**. It records each accuracy. The middle 95% of the accuracies is the bootstrap interval. This is the code from the lesson's Python box, with the same seed.

> **Predict.** How close is the bootstrap interval to the formula interval, 88.9% to 95.1%? Then run the cell.

In [ ]:
random.seed(1)
accs = sorted(sum(random.choices(results_b, k=300)) / 300 for _ in range(2000))
print("bootstrap middle 95%:", round(100 * accs[50], 1), "to", round(100 * accs[1949], 1))

> **Check.** You should see `89.0 to 95.0`. The formula gives 88.9 to 95.1: almost the same. `accs[50]` and `accs[1949]` cut 50 accuracies (2.5%) from each end of the 2,000.

The lesson's explorer draws a histogram of the resampled accuracies. Here it is as text bars, one bar for each accuracy that came up. Run the cell. You should see the bars gather around 92%, from 86.0% to 96.7%.

In [ ]:
from collections import Counter
counts = Counter(round(100 * a, 1) for a in accs)
values = sorted(counts)
bars([f"{v:.1f}%" for v in values], [counts[v] for v in values], width=30)

In professional work, `scipy.stats.bootstrap` or a short numpy loop does the same.

## 7. Error bars on a leaderboard

The lesson's leaderboard has four made-up models on 500 benchmark questions. The next cell calculates each 95% interval with the formula from section 5. Run it. You should see the lesson's intervals: P 77.8 to 84.6, Q 76.9 to 83.9, R 75.0 to 82.2 and S 70.2 to 77.8.

In [ ]:
rows = []
for name, accuracy in [("P", 81.2), ("Q", 80.4), ("R", 78.6), ("S", 74.0)]:
    p = accuracy / 100
    half = 1.96 * math.sqrt(p * (1 - p) / 500)
    rows.append([name, accuracy, 100 * half, 100 * (p - half), 100 * (p + half)])
table(["Model", "Accuracy (%)", "± (points)", "Low (%)", "High (%)"], rows, digits=1)

> **Check.** Each interval is about ±3.5 points. The intervals of P, Q and R overlap almost completely, so their order is not supported by the data.

## 8. Guided practice: the interval for model A

Model A is correct on 273 of the same 300 tickets.

> **Your turn.** Replace each `...` with a calculation. Give `p_a` and `se_a` as decimals (for example 0.92), and the half-width and the interval in percentage points (for example 3.07). Then run the check cell.

In [ ]:
p_a = 273 / 300
se_a = math.sqrt(p_a * (1 - p_a) / 300)
half_a = 100 * 1.96 * se_a
low_a = 100 * p_a - half_a
high_a = 100 * p_a + half_a

Run the check cell. Each line says "Check passed." or what to look at again.

In [ ]:
expect("the accuracy p", p_a, 0.91, tol=0.0001)
expect("the SE", se_a, 0.0165, tol=0.0001)
expect("the half-width", half_a, 3.24, tol=0.01)
expect("the low end", low_a, 87.8, tol=0.05)
expect("the high end", high_a, 94.2, tol=0.05)

Compare with model B, 88.9% to 95.1%. The two intervals overlap almost completely. Lesson 3 compares the two models directly.

## 9. Independent variation: a larger test set

Mei builds a random test set of **1,200 tickets**, and model B is again 92.0% correct.

> **Your turn.** Calculate the SE and the half-width in points, and the interval. Then use the square-root rule: Omar wants ±0.75 points, about half of ±1.5 points. How many tickets does that need? Run the check cell.

In [ ]:
se_1200 = 100 * math.sqrt(0.92 * 0.08 / 1200)
half_1200 = 1.96 * se_1200
low_1200 = 92.0 - half_1200
high_1200 = 92.0 + half_1200
n_needed = 4 * 1200

In [ ]:
expect("the SE", se_1200, 0.78, tol=0.01)
expect("the half-width", half_1200, 1.53, tol=0.01)
expect("the low end", low_1200, 90.5, tol=0.05)
expect("the high end", high_1200, 93.5, tol=0.05)
expect("the tickets needed", n_needed, 4800)

The square-root rule gives an estimate. The exact half-width for 1,200 tickets is 1.53 points, a little more than 2 × 0.75. So the formula needs a few more tickets. Run the cell. You should see `5027`: about 5,000 tickets, four times the data either way.

In [ ]:
# Solve 1.96 × √(p(1 − p) / n) = 0.0075 for n.
print(math.ceil(0.92 * 0.08 * (1.96 / 0.0075) ** 2))

## 10. Failure case: an interval read as a promise about one week

The report says "92% ± 3.1 points". Omar promises: "Every week, at least 89% of tickets will go to the right team." But the interval estimates one long-run number. It is not a promise about one week.

The next cell makes 8 weeks of 1,200 tickets. Each ticket is correct with a chance of exactly 92%, so the model never changes. The data is made up, with a fixed seed.

> **Predict.** Will every week show exactly 92.0%? Then run the cell.

In [ ]:
random.seed(41)
weekly = []
for week in range(1, 9):
    correct = sum(random.random() < 0.92 for _ in range(1200))
    weekly.append(100 * correct / 1200)
    print(f"week {week}: {weekly[-1]:.1f}%")
print(f"lowest {min(weekly):.1f}%, highest {max(weekly):.1f}%")

> **Check.** You should see weekly results from 91.2% to 92.4%, with the same model. For one week of 1,200 tickets, the SE is 0.78 points, so chance alone can move a week by about ±1.5 points. A week with a different ticket mix, such as December, can move further: that is why Grace's team saw 87.6%.

**Fix:** write what the interval covers: "On tickets like our test set, we estimate the accuracy at 92%. Values from 89% to 95% fit the test result." Track a random sample of tickets each week, and expect the number to move.

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m05/estimate-uncertainty/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you drew repeated samples from a known population: about 95 of 100 intervals contained the true mean, and 4 times more tickets gave intervals half as wide. You calculated the 95% interval for the 12 Monday reply times (14.7 to 42.8 minutes), for model B (88.9% to 95.1%) and for model A (87.8% to 94.2%). The bootstrap gave almost the same interval for model B as the formula.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Compare alternatives](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m05/estimate-uncertainty/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.